In [1]:
import json
from typing import List
import numpy as np
import librosa


def load_audio(path: str, sr: int = 16000):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y.astype(np.float32), sr

def load_subset(subset_path: str) -> List[dict]:
    with open(subset_path, "r", encoding="utf-8") as f:
        subset = json.load(f)
    if not isinstance(subset, list):
        raise ValueError("Subset file must contain a JSON list.")
    return subset

subset_path = '../../dcase_subset.json'
subset_path = '../../MMAU_pro_subset.json'

subset = load_subset(subset_path)
# import random
# random.shuffle(subset)

In [2]:
import sys
sys.path.append('../../')
from tools import generate_tool_descriptions, TOOL_NAMES
from tools import (
    ClippingTool,
    HumanVoiceEnhanceTool,
    AmplitudeNormalizeTool,
    LoudnessNormalizeTool,
    DCOffsetRemovalTool,
    SpectralNormalizeTool,
    TrimSilenceTool,
    PreEmphasisTool,
    SourceSeparationTool,
    ExtractTargetTool,
    RemoveTargetTool,
    SuperResolutionTool,
    PitchShiftTool,
    TimeStretchTool
)

tool_set =     [
        ClippingTool,
        HumanVoiceEnhanceTool,
        # AmplitudeNormalizeTool,
        # LoudnessNormalizeTool,
        # DCOffsetRemovalTool,
        # SpectralNormalizeTool,
        # TrimSilenceTool,
        # PreEmphasisTool,
        # SourceSeparationTool,
        ExtractTargetTool,
        RemoveTargetTool,
        SuperResolutionTool,
        PitchShiftTool,
        TimeStretchTool
    ]

# tool_set = None

tools_description = generate_tool_descriptions(
    tool_set
)


In [3]:
def gen_tool_call(toolname, parameters):
    assert toolname in TOOL_NAMES, f"Tool {toolname} is not in the toolset."
    tool_instruction =\
        {
            "tool": toolname,
            "parameters": parameters
        }
    
    return tool_instruction


In [4]:
def gen_target_extraction_parameter(question, answer, audio_path, description):
    # This is a placeholder function. You should implement the logic to generate parameters based on the question, answer, and audio_path.
    # audio, sr = load_audio(audio_path)
    # audio_length = len(audio) / sr

    return {
        "audio_path": audio_path,
        "target_description": description,  # or "instrumental", etc., depending on the question
        # 'audio_begin': "00:00:00.000",
        # 'audio_end': f"00:{int(audio_length // 60):02d}:{int(audio_length % 60):02d}.00"
    }


def gen_target_removal_parameter(question, answer, audio_path, description):
    # This is a placeholder function. You should implement the logic to generate parameters based on the question, answer, and audio_path.
    # audio, sr = load_audio(audio_path)
    # audio_length = len(audio) / sr

    return {
        "audio_path": audio_path,
        "target_description": description,  # or "instrumental", etc., depending on the question
        # 'audio_begin': "00:00:00.000",
        # 'audio_end': f"00:{int(audio_length // 60):02d}:{int(audio_length % 60):02d}.00"
    }

def gen_basic_parameter(question, answer, audio_path):
    # This is a placeholder function. You should implement the logic to generate parameters based on the question, answer, and audio_path.
    # audio, sr = load_audio(audio_path)
    # audio_length = len(audio) / sr

    return {
        "audio_path": audio_path,
        # "algorithm": "wiener",  # or "wiener_filter", etc., depending on the question
        # 'audio_begin': "00:00:00.000",
        # 'audio_end': f"00:{int(audio_length // 60):02d}:{int(audio_length % 60):02d}.00"
    }


In [6]:
from tqdm import tqdm

denoised_audio_root = './MMAU_denoised/tool_outputs'
def process_audio_path(item):
    item_id = item['id']
    denoised_audio_path = f"{denoised_audio_root}/{item_id}/tool_result_denoise_.wav"
    return denoised_audio_path
    
    
output_path = '../../tool_schedules.json'
subset = load_subset(subset_path)
schedules = []

for item in tqdm(subset):
    item_tool_scudule = []
    
    question = item['question']
    answer = item['answer']
    # audio_path = item['audio_path']
    audio_path = process_audio_path(item)
    item_tool_scudule.append(gen_tool_call('time_stretch', {
        'audio_path': audio_path,
        'rate': 1.2
    }))

    schedules.append(
        (item, item_tool_scudule)
    )

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(schedules, f, indent=4, ensure_ascii=False)

100%|██████████| 2013/2013 [00:00<00:00, 845497.09it/s]

In [10]:
from tqdm import tqdm
output_path = '../../human_voice_enhance_schedules.json'
subset = load_subset(subset_path)
schedules = []

for item in tqdm(subset):
    item_tool_scudule = []
    
    question = item['question']
    answer = item['answer']
    audio_path = item['audio_path']
    for desc in [ 'background sound']:
        denoise_parameter = gen_human_voice_enhance_parameter(question, answer, audio_path)
        item_tool_scudule.append(gen_tool_call('human_voice_enhance', denoise_parameter))

    schedules.append(
        (item, item_tool_scudule)
    )

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(schedules, f, indent=4, ensure_ascii=False)

100%|██████████| 2013/2013 [00:00<00:00, 962838.86it/s]